In [1]:
!pip install -U bitsandbytes>=0.46.1 -q

In [2]:
"""
OLMo-2-7B LoRA Finetuning on TED Multilingual Dataset
Format:  grouped paired (en anchor + all targets, no repetition)
Changes vs previous run:
  - LoRA r=8, alpha=16  (less aggressive, reduces catastrophic forgetting)
  - MMLU eval subset monitored during training (stop before degradation)
  - Paired grouped format: en once at top, each target lang below
  - OLMo-2-1124-7B base model

SETUP:
  1. wandb.ai account + API key as Colab secret "WANDB_API_KEY"
  2. Google Drive mounted at /content/drive
"""

# ── Install deps ───────────────────────────────────────────────────────────────
# !pip install -q wandb transformers peft bitsandbytes datasets accelerate lm_eval

import json
import os
import torch
import wandb
import numpy as np
from collections import defaultdict
from datasets import IterableDataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_ID   = "allenai/OLMo-2-1124-7B"
JSONL_PATH = "/content/drive/MyDrive/UCL/SNLP/TED2025/multi_way.jsonl"

REQ_LANGS  = ["en", "de", "id", "pt", "ar", "bn", "sw", "es", "ru", "fr", "ja", "zh-cn"]
LANG_ORDER = REQ_LANGS

USE_TAGS         = False
CHUNK_TOKENS     = 1024
MAX_LENGTH       = 1024
MIN_LANGS_PER_ROW  = 1    # accept any row with >= 1 of REQ_LANGS
MIN_LANGS_PER_TALK = 2    # skip entire talks with < 2 langs across all rows
MMLU_EVAL_SAMPLES = 100   # down from 400 — still statistically meaningful
MAX_CHUNKS = 40000

NUM_EPOCHS = 1            # watch eval loss — stop early if MMLU degrades

CKPT_DIR   = "/content/drive/MyDrive/UCL/SNLP/olmo2-translation-lora-checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/UCL/SNLP/olmo2-translation-lora-final"

WANDB_PROJECT = "olmo-translation-mmlu"
WANDB_RUN     = "olmo2-paired-r8"

# ── W&B Login ──────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    print("W&B logged in via Colab secret.")
except Exception:
    wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gbburgess (gbburgess-ucl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logged in via Colab secret.


In [3]:
# ── 1. Tokenizer ───────────────────────────────────────────────────────────────
# Must load before dataset — format_segment needs eos_token
print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ── 2. Dataset helpers ─────────────────────────────────────────────────────────
def nonempty_str(x) -> bool:
    return isinstance(x, str) and len(x.strip()) > 0

def prune_to_selected_langs(para_data, selected_langs):
    pd = para_data or {}
    return {l: pd[l] for l in selected_langs if nonempty_str(pd.get(l))}

def eligible_talk_ids(path, selected_langs, k=2):
    """Pre-scan: keep only talks that have >= k selected langs across all rows."""
    talk_to_langs = defaultdict(set)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            tid = obj.get("talk_id")
            if not tid:
                continue
            pd = prune_to_selected_langs(obj.get("para_data", {}), selected_langs)
            talk_to_langs[tid].update(pd.keys())
    return {tid for tid, langs in talk_to_langs.items() if len(langs) >= k}




def talk_chunk_generator(eligible_talks):
    """
    Pack rows within a talk into chunks up to CHUNK_TOKENS.
    Flushes at talk boundaries. Uses real tokenizer for accurate counts.
    """
    buffer_text  = ""
    buffer_tok   = 0
    current_talk = None

    def flush():
        nonlocal buffer_text, buffer_tok
        if buffer_text:
            yield {"text": buffer_text}
        buffer_text, buffer_tok = "", 0

    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj     = json.loads(line)
            talk_id = obj.get("talk_id")

            if not talk_id or talk_id not in eligible_talks:
                continue

            pd_sel = prune_to_selected_langs(obj.get("para_data", {}), REQ_LANGS)
            if len(pd_sel) < MIN_LANGS_PER_ROW:
                continue

            seg = format_segment(pd_sel)
            if not seg:
                continue

            seg_tok = len(seg) // 4

            if current_talk is None:
                current_talk = talk_id
            elif talk_id != current_talk:
                yield from flush()
                current_talk = talk_id

            if seg_tok >= CHUNK_TOKENS:
                yield from flush()
                yield {"text": seg}
            elif buffer_tok + seg_tok > CHUNK_TOKENS:
                yield from flush()
                buffer_text = seg
                buffer_tok  = seg_tok
            else:
                buffer_text += seg
                buffer_tok  += seg_tok

    yield from flush()


Loading tokenizer for allenai/OLMo-2-1124-7B...


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [4]:
def format_segment(para_data):
    """
    Individual paired format — one en→X pair per line, separated by blank lines.
    English is repeated as the source for each target language.

      en: {src_sent}
      de: {tgt_sent}

      en: {src_sent}
      es: {tgt_sent}

      en: {src_sent}
      zh-cn: {tgt_sent}
    """
    pd = prune_to_selected_langs(para_data, LANG_ORDER)
    if not pd:
        return ""

    langs    = [l for l in LANG_ORDER if l in pd]
    src_lang = "en" if "en" in pd else langs[0]
    src_text = pd[src_lang]

    pairs = []
    for tgt_lang in langs:
        if tgt_lang != src_lang:
            pairs.append(f"{src_lang}: {src_text}\n{tgt_lang}: {pd[tgt_lang]}")

    if not pairs:
        return f"{src_lang}: {src_text}" + tokenizer.eos_token

    return "\n\n".join(pairs) + tokenizer.eos_token



    # def format_segment(para_data):
    # """
    # Grouped paired format — English anchor once, all target languages below.
    # No repetition of English per pair.

    #   en: {src_sent}
    #   de: {tgt_sent}
    #   es: {tgt_sent}
    #   zh-cn: {tgt_sent}
    #   ...
    # """
    # pd = prune_to_selected_langs(para_data, LANG_ORDER)
    # if not pd:
    #     return ""

    # langs = [l for l in LANG_ORDER if l in pd]
    # if not langs:
    #     return ""

    # src_lang = "en" if "en" in pd else langs[0]
    # parts    = [f"{src_lang}: {pd[src_lang]}"]

    # for tgt_lang in langs:
    #     if tgt_lang != src_lang:
    #         parts.append(f"{tgt_lang}: {pd[tgt_lang]}")

    # if len(parts) == 1:
    #     # Only one language in this row — still useful signal
    #     pass

    # return "\n".join(parts) + tokenizer.eos_token

In [5]:

# Pre-scan eligible talks
print("Pre-scanning JSONL for eligible talks...")
eligible = eligible_talk_ids(JSONL_PATH, REQ_LANGS, k=MIN_LANGS_PER_TALK)
print(f"Eligible talks: {len(eligible)}")

# Build IterableDataset
# train_ds = IterableDataset.from_generator(
#     lambda: talk_chunk_generator(eligible)
# )
train_ds = IterableDataset.from_generator(
    lambda: talk_chunk_generator(eligible)
).take(MAX_CHUNKS)

# ── 3. Tokenize ────────────────────────────────────────────────────────────────
def tokenize(element):
    return tokenizer(element["text"], truncation=True, max_length=MAX_LENGTH)

print("Applying tokenizer to packed dataset...")
tokenized = train_ds.map(tokenize, remove_columns=["text"])

# Count chunks for max_steps (fast — no GPU)
# Replace the counting block with this:
print("Counting chunks for scheduler (one-time pass)...")
num_chunks = 0
buffer_tok = 0
current_talk = None

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        talk_id = obj.get("talk_id")
        if not talk_id or talk_id not in eligible:
            continue
        pd_sel = prune_to_selected_langs(obj.get("para_data", {}), REQ_LANGS)
        if len(pd_sel) < MIN_LANGS_PER_ROW:
            continue
        seg = format_segment(pd_sel)
        if not seg:
            continue
        seg_tok = len(seg) // 4  # char-based estimate, no tokenizer call
        if current_talk is None:
            current_talk = talk_id
        elif talk_id != current_talk:
            if buffer_tok > 0:
                num_chunks += 1
            buffer_tok = 0
            current_talk = talk_id
        if seg_tok >= CHUNK_TOKENS:
            if buffer_tok > 0:
                num_chunks += 1
                buffer_tok = 0
            num_chunks += 1
        elif buffer_tok + seg_tok > CHUNK_TOKENS:
            num_chunks += 1
            buffer_tok = seg_tok
        else:
            buffer_tok += seg_tok
if buffer_tok > 0:
    num_chunks += 1

effective_batch = 8 * 4
num_chunks  = min(num_chunks, MAX_CHUNKS)
max_steps   = (num_chunks * NUM_EPOCHS) // effective_batch
print(f"Chunks: {num_chunks} | Effective batch: {effective_batch} | max_steps: {max_steps}")


# ── 5. W&B init ────────────────────────────────────────────────────────────────
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "model":              MODEL_ID,
        "format":             "grouped-paired",
        "languages":          REQ_LANGS,
        "total_chunks":       num_chunks,
        "num_epochs":         NUM_EPOCHS,
        "max_steps":          max_steps,
        "chunk_tokens":       CHUNK_TOKENS,
        "max_length":         MAX_LENGTH,
        "lora_r":             8,
        "lora_alpha":         16,
        "lr":                 5e-5,
        "batch_size":         8,
        "grad_accum":         4,
        "quantization":       "4bit-nf4",
        "grad_checkpointing": True,
    },
)


Pre-scanning JSONL for eligible talks...
Eligible talks: 9104
Applying tokenizer to packed dataset...
Counting chunks for scheduler (one-time pass)...
Chunks: 40000 | Effective batch: 32 | max_steps: 1250


In [6]:
# ── 6. Quantization ────────────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ── 7. Load Base Model ─────────────────────────────────────────────────────────
if "model" in globals() and isinstance(globals()["model"], PeftModel):
    del globals()["model"]
    torch.cuda.empty_cache()

print(f"\nLoading {MODEL_ID} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

# ── 8. LoRA — r=8, alpha=16 ───────────────────────────────────────────────────
# Lower rank + matching alpha vs previous run (r=16, alpha=32).
# Reduces the capacity of the adapter to diverge from the base model,
# which should preserve MMLU performance while still learning translation alignment.
peft_config = LoraConfig(
    r=8,           # halved from 16
    lora_alpha=16, # matches r — effective scale factor = alpha/r = 1.0
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
wandb.log({
    "model/trainable_params": trainable,
    "model/trainable_pct":    100 * trainable / total,
})



# ── 10. Training Arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    max_steps=max_steps,
    logging_steps=10,
    bf16=True,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=False,   # we monitor manually via callback
    dataloader_num_workers=2,
    report_to="wandb",
)



Loading allenai/OLMo-2-1124-7B in 4-bit...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 8,388,608 || all params: 7,307,005,952 || trainable%: 0.1148


In [7]:
# ── 11. Trainer ────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer, mlm=False, pad_to_multiple_of=8
    ),
)

print(f"\nStarting OLMo-2 run:")
print(f"  Format: grouped-paired  |  LoRA r=8, alpha=16")
print(f"  {num_chunks} chunks | max_steps={max_steps}")
print(f"  Checkpointing every 200 steps → {CKPT_DIR}")
print(f"  Track live: https://wandb.ai/gbburgess-ucl/{WANDB_PROJECT}\n")

trainer.train()

# ── 12. Save ───────────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

wandb.summary["final_train_loss"] = trainer.state.log_history[-1].get("train_loss", None)
wandb.finish()
print(f"\nDone. Adapter saved to {OUTPUT_DIR}")



Starting OLMo-2 run:
  Format: grouped-paired  |  LoRA r=8, alpha=16
  40000 chunks | max_steps=1250
  Checkpointing every 200 steps → /content/drive/MyDrive/UCL/SNLP/olmo2-translation-lora-checkpoints
  Track live: https://wandb.ai/gbburgess-ucl/olmo-translation-mmlu



Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.


Step,Training Loss
10,1.223198
20,1.133658
30,1.138566
40,1.253819
50,1.208917
60,1.057281
70,1.123877
80,1.098442
90,1.081351
100,1.070202


KeyboardInterrupt: 

Sanity test


In [4]:
# ── Quick MMLU diagnostic — run this on the BASE model before any training ────
# Tells us: (1) is base OLMo-2 scoring correctly, (2) is our eval method right

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL_ID = "allenai/OLMo-2-1124-7B"
N        = 100

print("Loading base model for diagnostic...")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
model.config.use_cache = True   # ensure cache is ON for inference
model.eval()

# Check what token IDs A/B/C/D actually map to for this tokenizer
choice_tokens = {
    "A": tok.encode(" A", add_special_tokens=False)[0],  # 362
    "B": tok.encode(" B", add_special_tokens=False)[0],  # 426
    "C": tok.encode(" C", add_special_tokens=False)[0],  # 356
    "D": tok.encode(" D", add_special_tokens=False)[0],  # 423
}
print("Choice token IDs:", choice_tokens)

# Load MMLU
ds = load_dataset("cais/mmlu", "all", split="test").select(range(N))
answer_map = {0: "A", 1: "B", 2: "C", 3: "D"}

# Score " Option X" as a 2-token continuation
option_ids = {
    c: tok.encode(f" Option {c}", add_special_tokens=False)  # e.g. [' Option', ' A']
    for c in ["A", "B", "C", "D"]
}
print("Option token sequences:", option_ids)

correct = 0
for ex in ds:
    choices = ex["choices"]
    prompt  = (
        f"Question: {ex['question']}\n"
        f"A: {choices[0]}\nB: {choices[1]}\nC: {choices[2]}\nD: {choices[3]}\n"
        f"Answer:"
    )
    input_ids = tok(prompt, return_tensors="pt", truncation=True,
                    max_length=512).input_ids.to(model.device)

    scores = {}
    for c, cont_ids in option_ids.items():
        # Append continuation tokens and score their joint log-prob
        full_ids = torch.cat([
            input_ids,
            torch.tensor([cont_ids], device=model.device)
        ], dim=1)
        with torch.no_grad():
            out    = model(full_ids)
            logits = out.logits[0]  # (seq_len, vocab)

        # Score each continuation token at its position
        log_prob = 0
        for i, tok_id in enumerate(cont_ids):
            pos       = input_ids.shape[1] - 1 + i
            log_prob += torch.log_softmax(logits[pos], dim=-1)[tok_id].item()
        scores[c] = log_prob

    pred    = max(scores, key=scores.get)
    label   = answer_map[ex["answer"]]
    correct += int(pred == label)

acc = correct / len(ds)
print(f"Accuracy with ' Option X' scoring: {acc:.3f}")

Loading base model for diagnostic...


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

Choice token IDs: {'A': 362, 'B': 426, 'C': 356, 'D': 423}
Option token sequences: {'A': [7104, 362], 'B': [7104, 426], 'C': [7104, 356], 'D': [7104, 423]}
Accuracy with ' Option X' scoring: 0.330


In [5]:
BASE_MODEL = "allenai/OLMo-2-1124-7B"

for lang in ["en", "es", "zh", "de", "fr", "ar", "ru", "ja", "bn", "sw", "id", "pt"]:
    !lm_eval --model hf \
      --model_args "pretrained={BASE_MODEL},dtype=auto,device_map=auto" \
      --tasks "global_mmlu_full_{lang}" \
      --batch_size auto \
      --limit 0.1 \
      --output_path "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_baseline_olmo2/{lang}"

/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found
/bin/bash: line 1: lm_eval: command not found


In [8]:
!pip install -q lm_eval[vllm]


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 158.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━

In [10]:
!find / -path "*/lm_eval/tasks/global_mmlu*" -name "*.yaml" 2>/dev/null | head -5


/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default/sw/global_mmlu_sw_humanities.yaml
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default/sw/global_mmlu_sw_business.yaml
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default/sw/global_mmlu_sw_stem.yaml
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default/sw/global_mmlu_sw_medical.yaml
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default/sw/global_mmlu_sw_social_sciences.yaml


In [11]:
!find /usr/local/lib/python3.12/dist-packages/lm_eval/tasks/global_mmlu/default -name "*.yaml" -exec sed -i 's/test_split: test/test_split: dev/g' {} +

In [14]:
# Free the diagnostic model from GPU memory
del model
torch.cuda.empty_cache()
import gc; gc.collect()

# Verify memory is free
import torch
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU memory free: 22.5 GB


In [15]:
BASE_MODEL = "allenai/OLMo-2-1124-7B"


!lm_eval --model vllm \
  --model_args "pretrained=allenai/OLMo-2-1124-7B,dtype=auto,gpu_memory_utilization=0.75" \
  --tasks global_mmlu_full_en global_mmlu_full_es global_mmlu_full_fr global_mmlu_full_de \
          global_mmlu_full_id global_mmlu_full_pt global_mmlu_full_ru global_mmlu_full_zh \
          global_mmlu_full_ja global_mmlu_full_ar global_mmlu_full_sw global_mmlu_full_bn \
  --batch_size auto \
  --output_path "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_baseline_olmo2"

2026-03-09:18:44:33 INFO     [_cli.run:376] Selected Tasks: ['global_mmlu_full_en', 'global_mmlu_full_es', 'global_mmlu_full_fr', 'global_mmlu_full_de', 'global_mmlu_full_id', 'global_mmlu_full_pt', 'global_mmlu_full_ru', 'global_mmlu_full_zh', 'global_mmlu_full_ja', 'global_mmlu_full_ar', 'global_mmlu_full_sw', 'global_mmlu_full_bn']
2026-03-09:18:44:35 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-09:18:44:35 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': 'allenai/OLMo-2-1124-7B', 'dtype': 'auto', 'gpu_memory_utilization': 0.75}
2026-03-09 18:44:41.762623: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 18:44:41.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL_ID = "allenai/OLMo-2-1124-7B"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

ds = load_dataset("cais/mmlu", "all", split="test").select(range(5))
answer_map = {0: "A", 1: "B", 2: "C", 3: "D"}

# ── Diagnostic 1: what do A/B/C/D look like in this tokenizer? ────────────────
print("=== Token ID Diagnostic ===")
for c in ["A", "B", "C", "D"]:
    for variant in [c, f" {c}", f"{c}:", f"\n{c}", f"({c})"]:
        ids = tok.encode(variant, add_special_tokens=False)
        print(f"  {repr(variant):12} → {ids} → {[tok.decode([i]) for i in ids]}")

# ── Diagnostic 2: what does the model actually want to generate? ──────────────
print("\n=== Generation Diagnostic (first 3 examples) ===")
for ex in ds.select(range(3)):
    choices = ex["choices"]
    prompt  = (
        f"Question: {ex['question']}\n"
        f"A: {choices[0]}\nB: {choices[1]}\nC: {choices[2]}\nD: {choices[3]}\n"
        f"Answer:"
    )
    inputs  = tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    # See what the model freely generates
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    generated = tok.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Also show top 10 tokens at the Answer: position
    with torch.no_grad():
        out = model(**inputs)
    last     = out.logits[0, -1, :]
    top10    = torch.topk(last, 10)
    top_toks = [(tok.decode([i]), s.item()) for i, s in zip(top10.indices, top10.values)]

    correct = answer_map[ex["answer"]]
    print(f"\n  Q: {ex['question'][:60]}...")
    print(f"  Correct: {correct} | Generated: {repr(generated)}")
    print(f"  Top 10 tokens: {top_toks}")

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

=== Token ID Diagnostic ===
  'A'          → [32] → ['A']
  ' A'         → [362] → [' A']
  'A:'         → [32, 25] → ['A', ':']
  '\nA'        → [198, 32] → ['\n', 'A']
  '(A)'        → [4444, 8] → ['(A', ')']
  'B'          → [33] → ['B']
  ' B'         → [426] → [' B']
  'B:'         → [33, 25] → ['B', ':']
  '\nB'        → [198, 33] → ['\n', 'B']
  '(B)'        → [5462, 8] → ['(B', ')']
  'C'          → [34] → ['C']
  ' C'         → [356] → [' C']
  'C:'         → [34, 25] → ['C', ':']
  '\nC'        → [198, 34] → ['\n', 'C']
  '(C)'        → [3100, 8] → ['(C', ')']
  'D'          → [35] → ['D']
  ' D'         → [423] → [' D']
  'D:'         → [35, 25] → ['D', ':']
  '\nD'        → [198, 35] → ['\n', 'D']
  '(D)'        → [5549, 8] → ['(D', ')']

=== Generation Diagnostic (first 3 examples) ===

  Q: Find the degree for the given field extension Q(sqrt(2), sqr...
  Correct: B | Generated: ' Option B\n\nQuestion:'
  Top 10 tokens: [(' Option', -0.9765625), (' B', -1.1171875), (' C',